In [18]:
import faiss
import json
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
index = faiss.read_index("index/faiss.index")

with open("index/metadata.json", "r", encoding="utf-8") as f:
    metadata = json.load(f)

return model, index, metadata


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [ ]:
def retrieve(query, k=5):
    q_emb = model.encode([query], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q_emb, k)

    results = []
    for score, idx in zip(scores[0], ids[0]):
        m = metadata[idx]
        results.append({
            "score": float(score),
            "text": m["text"],
            "source": m["source"],
            "page": m["page"]
        })
    return results


## load prompt

In [7]:
from pathlib import Path

PROMPT_PATH = Path("prompts/clinical_rag_prompt.txt")

def load_prompt_template():
    return PROMPT_PATH.read_text(encoding="utf-8")


## Retrieve answer from prompt

In [9]:
def build_prompt(retrieved_chunks):
    """
    retrieved_chunks: list of dicts with keys:
      text, source, page
    """
    blocks = []

    for i, r in enumerate(retrieved_chunks, start=1):
        block = (
            f"[{i}] {r['source']}, p.{r['page']}\n"
            f"{r['text']}"
        )
        blocks.append(block)

    return "\n\n".join(blocks)


In [5]:
def call_llm(prompt):
    response = client.chat.completions.create(
        model="gpt-4.1-mini",   # or your chosen model
        messages=[
            {"role": "system", "content": "You follow the instructions strictly."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.0
    )
    return response.choices[0].message.content


In [12]:
q = "According to WHO guidelines, when should antiretroviral therapy be initiated?"


In [13]:
def rag_answer(question):
    retrieved = retrieve(question, k=6)

    # safety / confidence gate
    if not retrieved or retrieved[0]["score"] < 0.25:
        return "I don't have enough information in the guidelines."
    print(build_prompt(q, retrieved))
    prompt = build_prompt(question, retrieved)
    answer = call_llm(prompt)
    


    return answer


In [15]:
print(rag_answer(q))

NameError: name 'model' is not defined